In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

# DATASET
BASE = Path("/kaggle/input/datasets/vedantsaraswate/complete-backup-edu")

# Vale & Pereira (2017), CuGa1 parameters
a = 0.2
v = 0.057708

cities = [
    "ahmedabad", "bhubaneswar", "chennai", "hyderabad", "indore",
    "jaipur", "kolkata", "lucknow", "mumbai", "pune"
]

# LOAD ORIGINAL UDISE QUALITY SCORES
quality = pd.read_csv(BASE / "geo_data/outputs/final_education_scores.csv")
quality["city"] = quality["city"].str.strip().str.lower()
quality = quality[["city", "mean_Q7"]]

results = []

# CALCULATE CUMULATIVE-GAUSSIAN ACCESSIBILITY
for city in cities:

    data = np.load(BASE / "city_calculations" / f"{city}_calculations.npz")

    distances = data["distances_km"]
    population = data["population_15_19"]

    weights = np.where(
        distances <= a,
        1.0,
        np.exp(-((distances - a) ** 2) / v)
    )

    point_accessibility = weights.sum(axis=1)

    accessibility = np.average(
        point_accessibility,
        weights=population
    )

    results.append({
        "city": city,
        "Accessibility": accessibility
    })

# COMBINE ACCESSIBILITY AND QUALITY
results = pd.DataFrame(results)

results = results.merge(
    quality,
    on="city",
    how="left",
    validate="one_to_one"
)

assert results["mean_Q7"].notna().all(), "Missing UDISE quality scores!"

# FINAL EDUCATION INDEX
results["Education Index"] = (
    results["Accessibility"] * results["mean_Q7"]
)

results = results.sort_values(
    "Education Index", ascending=False
).reset_index(drop=True)

results.insert(0, "Rank", range(1, len(results) + 1))
results["City"] = results.pop("city").str.title()

# DISPLAY AND SAVE
print(results[[
    "Rank", "City", "Accessibility", "mean_Q7", "Education Index"
]].to_string(index=False))

results.to_csv(
    "/kaggle/working/cumulative_gaussian_education_index.csv",
    index=False
)

print("\nSaved final results to Kaggle working directory.")

 Rank        City  Accessibility  mean_Q7  Education Index
    1      Mumbai       1.147985 0.838885         0.963027
    2     Chennai       0.949989 0.899384         0.854405
    3      Indore       0.944299 0.866401         0.818142
    4   Hyderabad       0.570723 0.818623         0.467208
    5        Pune       0.530333 0.869941         0.461358
    6     Kolkata       0.450483 0.851860         0.383749
    7     Lucknow       0.343586 0.868626         0.298448
    8 Bhubaneswar       0.194507 0.856083         0.166514
    9      Jaipur       0.147691 0.856730         0.126532
   10   Ahmedabad       0.121027 0.877610         0.106215

Saved final results to Kaggle working directory.


In [2]:
from scipy.stats import spearmanr, kendalltau
import pandas as pd

# Original 2SFCA ranking
original = pd.read_csv(
    BASE / "geo_data/outputs/final_education_scores.csv"
)

# Cumulative-Gaussian ranking
cg = pd.read_csv(
    "/kaggle/working/cumulative_gaussian_education_index.csv"
)

# Match cities
comparison = original[["city", "Rank"]].merge(
    cg[["City", "Rank"]],
    left_on=original["city"].str.lower(),
    right_on=cg["City"].str.lower(),
    suffixes=("_2SFCA", "_CG")
)

# Calculate rank correlations
rho, p_value = spearmanr(
    comparison["Rank_2SFCA"],
    comparison["Rank_CG"]
)

tau, kendall_p = kendalltau(
    comparison["Rank_2SFCA"],
    comparison["Rank_CG"]
)

print("Spearman correlation:", round(rho, 4))
print("Spearman p-value:", round(p_value, 6))
print("Kendall's tau:", round(tau, 4))
print("Kendall p-value:", round(kendall_p, 6))

print("\nCity ranking comparison:")
print(comparison.to_string(index=False))

Spearman correlation: 0.9515
Spearman p-value: 2.3e-05
Kendall's tau: 0.8667
Kendall p-value: 0.000115

City ranking comparison:
      key_0        city  Rank_2SFCA        City  Rank_CG
    chennai     chennai           1     Chennai        2
     mumbai      mumbai           2      Mumbai        1
     indore      indore           3      Indore        3
  hyderabad   hyderabad           4   Hyderabad        4
       pune        pune           5        Pune        5
    kolkata     kolkata           6     Kolkata        6
bhubaneswar bhubaneswar           7 Bhubaneswar        8
     jaipur      jaipur           8      Jaipur        9
    lucknow     lucknow           9     Lucknow        7
  ahmedabad   ahmedabad          10   Ahmedabad       10
